###1. Basic Tasks

**1. Connect Power BI Desktop to a Databricks SQL warehouse via Partner Connect (or a manual
connection) and build one report against a gold table.**

![image_1789377930823.png](./image_1789377930823.png "image_1789377930823.png")

**2. Create a Unity Catalog connection to an external PostgreSQL (or MySQL) database and a foreign
catalog exposing one of its tables.**

**2. Unity Catalog Connection to External PostgreSQL**

I connected **Databricks Unity Catalog with a PostgreSQL database hosted on NeonDB** using a Unity Catalog connection and created a **foreign catalog** for the NeonDB database.

The table created in **NeonDB** was successfully exposed through the foreign catalog in Databricks and could be queried directly from Databricks without physically copying the table into a Delta table.

**Flow:**

```text
NeonDB (PostgreSQL)
       ↓
Unity Catalog Connection
       ↓
Foreign Catalog
       ↓
Databricks
       ↓
Query NeonDB table directly
```

For example, I was able to access the external table using:

```sql
SELECT *
FROM <foreign_catalog>.<schema>.<table>;
```

**Result:** The PostgreSQL table from NeonDB was successfully visible and queryable in Databricks through **Lakehouse Federation**, demonstrating access to external data without creating a separate physical copy in the Lakehouse.


**3. Read about Lakebase and write a short summary of when you'd reach for it instead of a Delta table.**

Lakebase is Databricks’ managed PostgreSQL database designed for operational/OLTP or transactional workloads, while Delta tables are primarily designed for analytical workloads. I’d reach for Lakebase instead of a Delta table when an application needs low-latency, frequent row-level reads/writes, transactions, user sessions, chat history, workflow state, or other operational data.

I’d keep using Delta tables for ETL/ELT pipelines, batch or streaming data, large analytical queries, BI dashboards, aggregations, and ML workloads. In many applications, the best architecture is to use Lakebase for the application's transactional state and Delta for the analytical system of record.


###2. Intermediate Tasks

**4. Publish your Power BI report to the Power BI service and document the two ways data could go stale
for this connection type (scheduled refresh vs. live query).**

After publishing the Power BI report to the Power BI Service, the data can become stale in two different ways depending on the connection mode:
1. Scheduled Refresh:
- If the report uses Import mode, Power BI stores a copy of the data in its semantic model. The data remains unchanged until the next scheduled/manual refresh, so the report can show stale data between refreshes. For example, if the source changes at 10:00 AM but the next refresh is at 12:00 PM, the report may continue showing the 10:00 AM data until the refresh runs.
2. Live Query:
- If the report uses DirectQuery/Live connection, queries are sent to the underlying data source when users interact with the report. Data can still appear stale if the source itself has not been updated, if query results are cached, or if the connection/query has latency or availability issues. Therefore, "live" does not necessarily mean every visual is guaranteed to reflect the source's latest committed data at every instant.

**5. Write a federated query that joins a native Unity Catalog Delta table with a foreign-catalog table
from the external Postgres database, and confirm no data was physically copied first.**

In [0]:
%sql
DESCRIBE EXTENDED dev.silver.customers_table;

In [0]:
%sql
DESCRIBE EXTENDED neon_db_connection_catalog.public.orders;

In [0]:
%sql
SELECT c.customer_id, c.name, c.email, o.order_id, o.order_date, o.amount, o.status
FROM dev.silver.customers_table c
INNER JOIN neon_db_connection_catalog.public.orders o
ON c.customer_id = o.customer_id;

![managed_table_1789404997023.png](./managed_table_1789404997023.png "managed_table_1789404997023.png")
![foreign_table_1789405007515.png](./foreign_table_1789405007515.png "foreign_table_1789405007515.png")

Created and successfully executed a federated query joining the native Unity Catalog Delta table `dev.silver.customers_table` with the external PostgreSQL `orders` table from NeonDB. The external table was accessed through the foreign catalog, and no physical copy of the PostgreSQL data was created in Delta before executing the JOIN.

**6. Set up a Databricks-to-Databricks OpenShare of one gold table with a partner workspace (or
simulate the recipient side) and confirm what content types (tables, views, volumes) are supported.**

In [0]:
%sql
CREATE SHARE IF NOT EXISTS customer_share;

In [0]:
%sql
SELECT * FROM dev.gold.top_customers

In [0]:
%sql
ALTER SHARE customer_share 
ADD TABLE dev.gold.top_customers

In [0]:
%sql
SELECT * FROM dev.gold.top_5_customer

In [0]:
%sql
ALTER SHARE customer_share 
ADD VIEW dev.gold.top_5_customer

![share_add_1789392416604.png](./share_add_1789392416604.png "share_add_1789392416604.png")

In [0]:
%sql
ALTER SHARE customer_share
ADD VOLUME temp.temp_schema.temp_volume;

I have performed the above

**1. Provider side:** Created customer_share and added the gold table, view and volumn successfully.

**2. Recipient side:** Cross-workspace recipient creation was attempted, but the Free Edition metastore does not have External Delta Sharing enabled. Therefore, the recipient side was simulated rather than actually connected.
![Screenshot 2026-09-14 at 4.34.04 PM_1789384014846.png](./Screenshot 2026-09-14 at 4.34.04 PM_1789384014846.png "Screenshot 2026-09-14 at 4.34.04 PM_1789384014846.png")

**7. (Data Analyst) Import an existing Power BI file into an AI/BI dashboard and note what did and didn't translate cleanly.**

![image-2_1789386646035.png](./image-2_1789386646035.png "image-2_1789386646035.png")
![databricks_dashboard_1789392155983.png](./databricks_dashboard_1789392155983.png "databricks_dashboard_1789392155983.png")
- The Power BI dashboard was compared with the corresponding Databricks AI/BI dashboard. Basic visual concepts and fields translated, but some formatting and visualization properties changed, including the light-to-dark theme, chart layout, axis labels, and pie-to-donut conversion. More importantly, the customer-segment visualization did not translate cleanly: Power BI shows Sum of CLV across Premium, Gold, Silver, and Regular segments, whereas the Databricks version shows Count of Unique customer_segment with only Returning Customer, indicating that the metric/filter configuration needs to be reviewed after import.

###3. Advanced Tasks

**8. Design a data-sharing decision matrix for Cyntexa: for a given partner scenario (has their own
Databricks workspace vs. doesn't; needs tables only vs. needs AI assets), determine which
OpenSharing protocol — Databricks-to-Databricks vs. Databricks-to-Open — applies and why.**

| Partner scenario                                | Requirement        | Recommended protocol                     | Why                                                                                                                                             |
| ----------------------------------------------- | ------------------ | ---------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------- |
| Partner **has their own Databricks workspace**  | Tables only        | **Databricks-to-Databricks OpenSharing** | Best choice for sharing Delta tables directly between Unity Catalog-enabled Databricks workspaces without requiring token-based authentication. |
| Partner **has their own Databricks workspace**  | Tables + AI assets | **Databricks-to-Databricks OpenSharing** | Supports tables plus supported AI/data assets such as models and other Databricks-native assets.                                                |
| Partner **doesn't have a Databricks workspace** | Tables only        | **Databricks-to-Open OpenSharing**       | Designed for external recipients using an open sharing protocol rather than another Databricks workspace.                                       |
| Partner **doesn't have a Databricks workspace** | AI assets          | **Databricks-to-Open**, with limitations | Use it for supported shareable assets, but Databricks-to-Open has a more limited set of supported asset types than Databricks-to-Databricks.    |

**Decision rule:**

If the partner has a Databricks workspace → use Databricks-to-Databricks OpenSharing. If they don't have a Databricks workspace → use Databricks-to-Open OpenSharing. When AI assets are required, prefer Databricks-to-Databricks because it supports a broader range of Databricks-native assets.

**9. Evaluate query federation vs. building a nightly copy pipeline for the Postgres source: under what
data-freshness and query-volume conditions does federation stop making sense?**

| Condition                   | Query Federation                         | Nightly Copy Pipeline                       |
| --------------------------- | ---------------------------------------- | ------------------------------------------- |
| **Data freshness required** | Minutes/near real-time                   | Daily or hourly freshness is acceptable     |
| **Query volume**            | Low to moderate                          | High/repeated queries                       |
| **Queries**                 | Occasional ad-hoc queries                | Frequent BI dashboards, reports, analytics  |
| **Source PostgreSQL load**  | Source can handle the queries            | Source should not be repeatedly queried     |
| **Performance requirement** | Some network/query latency is acceptable | Consistent, fast query performance required |
| **Data size**               | Small/moderate datasets                  | Large datasets queried repeatedly           |
| **Best use case**           | Operational/near-real-time lookup        | Analytics and BI workloads                  |

Federation starts becoming less attractive when:
1. The business does not need fresh data:
- If users only need data updated once per night, continuously querying PostgreSQL provides little benefit. A nightly ETL/ELT copy into Delta is usually more appropriate.
2. Query volume becomes high:
- If hundreds or thousands of dashboard users/queries repeatedly hit the same PostgreSQL tables, federation can create network latency and additional load on the source database. Copying the data to Delta allows Databricks to serve those analytical queries locally.
3. Complex analytical queries become common:
- Large joins, aggregations, and repeated scans are generally better suited to Delta tables than an operational PostgreSQL database.
4. PostgreSQL becomes a performance bottleneck:
- If federated queries start affecting the source application's performance, move the analytical workload to Delta.


**10. Propose an LTAP architecture for a new Cyntexa feature (e.g., a real-time inventory-check app) that
needs both OLTP writes (Lakebase) and OLAP analytics (Lakehouse) on the same data, specifying
what syncs where and who owns each side operationally.**

**LTAP Architecture – Cyntexa Real-Time Inventory App**

For a real-time inventory application, I would use **Lakebase for OLTP** and the **Lakehouse for OLAP analytics**.

* **Lakebase:** Acts as the operational system of record. The inventory application performs real-time **INSERT, UPDATE, DELETE, and read operations** here. The Application/Backend team owns the Lakebase database and its transactional workload.
* **Lakehouse:** Uses the inventory data for **reporting, dashboards, trend analysis, forecasting, and ML**. The Data Engineering/Analytics team owns the analytical layer.
* **Data synchronization:** Changes can flow from **Lakebase → Lakehouse** through **Lakehouse//RT** for near-real-time analytics or **Lakebase Change Data Feed (CDF) → Delta tables** when persistent analytical datasets are needed.
* **Ownership:** The application team is responsible for the operational Lakebase workload, while the Data Engineering/Analytics team manages the Lakehouse, pipelines, and analytical use cases.
* **Key principle:** Lakebase should be the **single writer/source of truth for operational inventory**, while the Lakehouse is primarily used for **analytical consumption**, avoiding conflicting writes between the two systems.
